# OffSide 2026 — Football Analytics Datathon
**Objective:** Predict `scored_flag` (probability player scores ≥1 goal)  
**Metric:** Average Precision (AP)  
**Key design decisions:**
- GroupKFold by player_id (not StratifiedKFold) — prevents player-level leakage
- home/away club goals kept (available in test, not pure leakage)
- LightGBM + CatBoost ensemble
- OOF target encoding for player/club/position aggregates
- 15 Optuna trials max — feature engineering beats hyperparameter search

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
import optuna
import warnings
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
N_TRIALS = 15   # feature engineering >> hyperparameter tuning in a 12hr datathon

np.random.seed(SEED)
print('Setup complete')

## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

# Extract player_id and match_id from appearance_id (format: playerid_matchid)
for df in [train, test]:
    df['player_id'] = df['appearance_id'].str.split('_').str[0]
    df['match_id']  = df['appearance_id'].str.split('_').str[1]

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Unique players in train: {train["player_id"].nunique():,}')
print(f'Avg appearances per player: {len(train)/train["player_id"].nunique():.1f}')
print(f'\nTarget distribution:')
print(train['scored_flag'].value_counts())
print(f'Scoring rate: {train["scored_flag"].mean()*100:.2f}%')

# Confirm club goals columns are in test (they are — not dropping them)
leakage_candidates = ['home_club_goals', 'away_club_goals', 'goal_diff_abs']
print(f'\nClub goal columns in test: {[c for c in leakage_candidates if c in test.columns]}')
print('→ Keeping these — they are in test and are match-context features, not pure leakage')
print('  (home_club_goals=0 means player definitely scored 0; provides upper bound signal)')

## 3. EDA (fast — 20 minutes max)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

# Scoring rate by sub_position
sub_rate = train.groupby('sub_position')['scored_flag'].mean().sort_values(ascending=True)
sub_rate.plot(kind='barh', ax=axes[0], color='#378ADD')
axes[0].set_title('Scoring rate by sub_position')

# Scoring rate by market tier
tier_order = ['LOW','MEDIUM','HIGH','ELITE']
tier_rate = train.groupby('market_value_tier')['scored_flag'].mean().reindex(tier_order)
tier_rate.plot(kind='bar', ax=axes[1], color='#1D9E75')
axes[1].set_title('Scoring rate by market tier')
axes[1].tick_params(axis='x', rotation=0)

# Home vs away
train.groupby('home_away')['scored_flag'].mean().plot(kind='bar', ax=axes[2], color='#EF9F27')
axes[2].set_title('Home vs away scoring rate')
axes[2].tick_params(axis='x', rotation=0)

# xG distribution for scorers vs non-scorers
has_xg = train.dropna(subset=['avg_xG'])
axes[3].hist(has_xg[has_xg['scored_flag']==False]['avg_xG'], bins=40, alpha=0.6,
             label='Did not score', color='#5DCAA5', density=True)
axes[3].hist(has_xg[has_xg['scored_flag']==True]['avg_xG'], bins=40, alpha=0.6,
             label='Scored', color='#D85A30', density=True)
axes[3].set_title('avg_xG distribution by target')
axes[3].legend(fontsize=9)

# Minutes played distribution
axes[4].hist(train[train['scored_flag']==False]['minutes_played'], bins=40, alpha=0.6,
             label='Did not score', color='#5DCAA5', density=True)
axes[4].hist(train[train['scored_flag']==True]['minutes_played'], bins=40, alpha=0.6,
             label='Scored', color='#D85A30', density=True)
axes[4].set_title('Minutes played distribution by target')
axes[4].legend(fontsize=9)

# Scoring rate by season (trend check)
season_rate = train.groupby('season')['scored_flag'].mean()
season_rate.plot(ax=axes[5], marker='o', color='#7F77DD')
axes[5].set_title('Scoring rate by season')

plt.tight_layout()
plt.savefig('eda.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Validation Strategy — GroupKFold by Player

In [ ]:
# CRITICAL: Same player appears ~39 times on average.
# StratifiedKFold puts the same player in both train and val folds.
# Model memorises player-level features → OOF AP is artificially inflated.
#
# GroupKFold ensures every player's rows are EITHER in train OR in val.
# This gives a honest estimate of generalisation to unseen players.
#
# Note: GroupKFold does NOT stratify, so fold class balance may vary slightly.
# This is acceptable — AP is robust to small class imbalance variation.

groups = train['player_id'].values
print(f'Unique player groups: {len(np.unique(groups)):,}')
print(f'GroupKFold with {N_FOLDS} folds → each fold holds out ~{len(np.unique(groups))//N_FOLDS:,} players')

# Sanity check: what % of test players also appear in train?
train_players = set(train['player_id'].unique())
test_players  = set(test['player_id'].unique())
overlap_pct   = len(train_players & test_players) / len(test_players) * 100
print(f'\nTest players also in train: {overlap_pct:.1f}%')
if overlap_pct > 50:
    print('→ High overlap: player-level aggregate features will be very useful')
else:
    print('→ Lower overlap: position/club aggregates matter more than player-level')

## 5. OOF Target Encoding for Aggregate Features

In [ ]:
# Target-encoded aggregates are powerful but MUST be computed OOF to avoid leakage.
# For test rows: use full train statistics.

GLOBAL_MEAN = train['scored_flag'].mean()
SMOOTH_K    = 20  # smoothing factor — higher = more shrinkage toward global mean

def smooth_target_encode(train_df, test_df, group_col, target='scored_flag', k=SMOOTH_K):
    """Smoothed mean target encoding. OOF for train, full-train for test."""
    stats = train_df.groupby(group_col)[target].agg(['mean','count']).reset_index()
    stats.columns = [group_col, f'{group_col}_mean', f'{group_col}_count']
    stats[f'{group_col}_enc'] = (
        stats[f'{group_col}_count'] * stats[f'{group_col}_mean'] + k * GLOBAL_MEAN
    ) / (stats[f'{group_col}_count'] + k)
    test_merged = test_df[[group_col]].merge(stats[[group_col, f'{group_col}_enc']], on=group_col, how='left')
    test_encoded = test_merged[f'{group_col}_enc'].fillna(GLOBAL_MEAN).values
    return stats, test_encoded


def compute_oof_target_encodings(train_df, test_df, group_cols, n_folds=5, seed=42):
    """Compute OOF target encodings for multiple group columns."""
    oof_encodings  = {c: np.zeros(len(train_df)) for c in group_cols}
    test_encodings = {c: np.zeros(len(test_df))  for c in group_cols}

    gkf    = GroupKFold(n_splits=n_folds)
    groups = train_df['player_id'].values

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
        tr_fold  = train_df.iloc[tr_idx]
        val_fold = train_df.iloc[val_idx]

        for col in group_cols:
            stats, _ = smooth_target_encode(tr_fold, val_fold, col)
            merged   = val_fold[[col]].merge(
                stats[[col, f'{col}_enc']], on=col, how='left'
            )
            oof_encodings[col][val_idx] = merged[f'{col}_enc'].fillna(GLOBAL_MEAN).values

    # For test: use full train
    for col in group_cols:
        _, test_encoded = smooth_target_encode(train_df, test_df, col)
        test_encodings[col] = test_encoded

    return oof_encodings, test_encodings


ENCODE_COLS = ['player_id', 'sub_position', 'home_club_name', 'away_club_name',
               'country_name', 'competition_type', 'market_value_tier']
ENCODE_COLS = [c for c in ENCODE_COLS if c in train.columns]

print('Computing OOF target encodings for:', ENCODE_COLS)
oof_enc, test_enc = compute_oof_target_encodings(train, test, ENCODE_COLS, N_FOLDS, SEED)

for col in ENCODE_COLS:
    train[f'te_{col}'] = oof_enc[col]
    test[f'te_{col}']  = test_enc[col]

print('Done. New columns:', [f'te_{c}' for c in ENCODE_COLS])
print(f'\nSanity check — te_player_id mean: {train["te_player_id"].mean():.4f} (≈ global mean {GLOBAL_MEAN:.4f})')

## 6. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()

    # --- Drop identifier / metadata cols ---
    drop_cols = ['appearance_id', 'date', 'home_club_id', 'away_club_id',
                 'stadium', 'referee', 'name_x', 'name_y', 'player_name',
                 'match_id']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # --- Position-aware xG interactions (key signals) ---
    df['xG_x_attacker']   = df['avg_xG'].fillna(0) * df['is_attacker'].astype(float)
    df['xG_x_finisher']   = df['avg_xG'].fillna(0) * df['finisher_flag'].astype(float)
    df['xG_x_minutes']    = df['avg_xG'].fillna(0) * df['minutes_ratio']
    df['npxG_x_minutes']  = df['avg_npxG'].fillna(0) * df['minutes_ratio']
    df['xG_per_shot']     = df['avg_xG'] / (df['avg_shots'] + 1e-6)
    df['shot_x_minutes']  = df['avg_shots'].fillna(0) * df['minutes_ratio']

    # --- Starter × attacker (high-value interaction) ---
    df['starter_attacker'] = df['starter_flag'].astype(float) * df['is_attacker'].astype(float)
    df['starter_finisher'] = df['starter_flag'].astype(float) * df['finisher_flag'].astype(float)
    df['prime_attacker']   = df['prime_age_flag'].astype(float) * df['is_attacker'].astype(float)

    # --- Market value interactions ---
    df['mv_x_attacker']   = df['log_market_value'].fillna(0) * df['is_attacker'].astype(float)
    df['mv_x_xG']         = df['log_market_value'].fillna(0) * df['avg_xG'].fillna(0)
    df['mv_x_minutes']    = df['log_market_value'].fillna(0) * df['minutes_ratio']
    df['elite_attacker']  = ((df['market_value_tier'] == 'ELITE') & df['is_attacker']).astype(float)

    # --- International goal signal ---
    df['intl_goal_rate']  = df['international_goals'] / (df['international_caps'] + 1e-6)
    df['intl_goal_rate']  = df['intl_goal_rate'].fillna(0)

    # --- Team goal context (not leakage — in test too) ---
    df['team_goals']      = np.where(df['home_away'] == 'HOME',
                                     df['home_club_goals'], df['away_club_goals'])
    df['opponent_goals']  = np.where(df['home_away'] == 'HOME',
                                     df['away_club_goals'], df['home_club_goals'])
    df['team_scored']     = (df['team_goals'] > 0).astype(int)
    df['high_scoring']    = (df['team_goals'] >= 3).astype(int)
    df['winning']         = (df['team_goals'] > df['opponent_goals']).astype(int)

    # --- Minutes buckets ---
    df['played_over_60']  = (df['minutes_played'] >= 60).astype(int)
    df['is_impact_sub']   = (df['substitute_flag'] & (df['minutes_played'] >= 30)).astype(int)

    # --- Season (already int) ---
    df['season'] = df['season'].astype(int)

    # --- Encode categoricals for LightGBM ---
    cat_cols = ['home_away', 'competition_type', 'confederation', 'market_value_tier',
                'position', 'sub_position', 'foot', 'age_bucket', 'country_name',
                'home_club_name', 'away_club_name', 'country_of_citizenship']
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype('category')

    return df


train_fe = engineer_features(train)
test_fe  = engineer_features(test)

print(f'Train after FE: {train_fe.shape}')
print(f'Test after FE:  {test_fe.shape}')

## 7. Prepare Model Inputs

In [ ]:
TARGET = 'scored_flag'

DROP_FROM_X = [TARGET, 'player_id']

feature_cols = [c for c in train_fe.columns
                if c not in DROP_FROM_X and c in test_fe.columns]

X      = train_fe[feature_cols]
y      = train_fe[TARGET].astype(int)
groups = train_fe['player_id'].values
X_test = test_fe[feature_cols]
test_ids = test['appearance_id']

cat_features_lgb = [c for c in feature_cols if X[c].dtype.name == 'category']
cat_features_cb  = [feature_cols.index(c) for c in cat_features_lgb]

SPW = (y == 0).sum() / (y == 1).sum()

print(f'Features: {len(feature_cols)}')
print(f'Categorical features: {len(cat_features_lgb)}')
print(f'X shape: {X.shape} | X_test: {X_test.shape}')
print(f'Positive rate: {y.mean()*100:.2f}%')
print(f'scale_pos_weight: {SPW:.2f}')

## 8. GroupKFold Validation — LightGBM Baseline

In [ ]:
def run_lgbm(X, y, groups, X_test, params, n_folds=5, seed=42):
    """GroupKFold LightGBM. Returns OOF and test preds."""
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    ap_scores  = []

    gkf = GroupKFold(n_splits=n_folds)

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, random_state=seed, verbose=-1)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )

        val_prob = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_prob
        test_preds += model.predict_proba(X_test)[:, 1] / n_folds

        ap = average_precision_score(y_val, val_prob)
        ap_scores.append(ap)
        print(f'  Fold {fold+1}  AP={ap:.4f}  best_iter={model.best_iteration_}  '
              f'val_pos_rate={y_val.mean()*100:.1f}%')

    oof_ap = average_precision_score(y, oof_preds)
    print(f'\n  OOF AP: {oof_ap:.4f}  (mean fold: {np.mean(ap_scores):.4f} ± {np.std(ap_scores):.4f})')
    return oof_preds, test_preds, oof_ap


lgb_baseline_params = {
    'objective'          : 'binary',
    'metric'             : 'average_precision',
    'n_estimators'       : 2000,
    'learning_rate'      : 0.05,
    'num_leaves'         : 63,
    'min_child_samples'  : 50,
    'subsample'          : 0.8,
    'colsample_bytree'   : 0.8,
    'reg_alpha'          : 0.1,
    'reg_lambda'         : 1.0,
    'scale_pos_weight'   : SPW,
    'n_jobs'             : -1,
}

print('=== LIGHTGBM BASELINE (GroupKFold) ===')
oof_lgb_base, test_lgb_base, ap_lgb_base = run_lgbm(
    X, y, groups, X_test, lgb_baseline_params, N_FOLDS, SEED
)

## 9. CatBoost Baseline

In [ ]:
def run_catboost(X, y, groups, X_test, params, n_folds=5, seed=42):
    """GroupKFold CatBoost. Native categorical handling — no encoding needed."""
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    ap_scores  = []

    cat_cols = [c for c in X.columns if X[c].dtype.name == 'category']
    X_cb      = X.copy()
    X_test_cb = X_test.copy()
    for c in cat_cols:
        X_cb[c]      = X_cb[c].astype(str).replace('nan', 'MISSING')
        X_test_cb[c] = X_test_cb[c].astype(str).replace('nan', 'MISSING')

    gkf = GroupKFold(n_splits=n_folds)

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_cb, y, groups=groups)):
        X_tr, X_val = X_cb.iloc[tr_idx], X_cb.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        pool_tr  = cb.Pool(X_tr,  y_tr,  cat_features=cat_cols)
        pool_val = cb.Pool(X_val, y_val, cat_features=cat_cols)

        model = cb.CatBoostClassifier(**params, random_seed=seed)
        model.fit(pool_tr, eval_set=pool_val, verbose=False)

        val_prob = model.predict_proba(pool_val)[:, 1]
        oof_preds[val_idx] = val_prob

        pool_test = cb.Pool(X_test_cb, cat_features=cat_cols)
        test_preds += model.predict_proba(pool_test)[:, 1] / n_folds

        ap = average_precision_score(y_val, val_prob)
        ap_scores.append(ap)
        print(f'  Fold {fold+1}  AP={ap:.4f}  best_iter={model.best_iteration_}')

    oof_ap = average_precision_score(y, oof_preds)
    print(f'\n  OOF AP: {oof_ap:.4f}  (mean fold: {np.mean(ap_scores):.4f} ± {np.std(ap_scores):.4f})')
    return oof_preds, test_preds, oof_ap


cb_baseline_params = {
    'iterations'         : 2000,
    'learning_rate'      : 0.05,
    'depth'              : 6,
    'l2_leaf_reg'        : 3,
    'eval_metric'        : 'PRAUC',
    'early_stopping_rounds': 100,
    'loss_function'      : 'Logloss',
    'auto_class_weights' : 'Balanced',
    'task_type'          : 'CPU',
}

print('=== CATBOOST BASELINE (GroupKFold) ===')
oof_cb_base, test_cb_base, ap_cb_base = run_catboost(
    X, y, groups, X_test, cb_baseline_params, N_FOLDS, SEED
)

print(f'\nLightGBM baseline OOF AP: {ap_lgb_base:.4f}')
print(f'CatBoost baseline OOF AP: {ap_cb_base:.4f}')

## 10. Optuna Tuning (15 trials — best model only)

In [ ]:
best_baseline_model = 'lgb' if ap_lgb_base >= ap_cb_base else 'catboost'
print(f'Tuning: {best_baseline_model}')

def objective_lgb(trial):
    params = {
        'objective'        : 'binary',
        'metric'           : 'average_precision',
        'n_estimators'     : 2000,
        'n_jobs'           : -1,
        'scale_pos_weight' : SPW,
        'verbose'          : -1,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    gkf = GroupKFold(n_splits=3)
    scores = []
    for tr_idx, val_idx in gkf.split(X, y, groups=groups):
        m = lgb.LGBMClassifier(**params, random_state=SEED)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        scores.append(average_precision_score(y.iloc[val_idx], m.predict_proba(X.iloc[val_idx])[:,1]))
    return np.mean(scores)


def objective_cb(trial):
    params = {
        'iterations'           : 2000,
        'eval_metric'          : 'PRAUC',
        'early_stopping_rounds': 50,
        'loss_function'        : 'Logloss',
        'auto_class_weights'   : 'Balanced',
        'task_type'            : 'CPU',
        'verbose'              : False,
        'learning_rate'        : trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'depth'                : trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg'          : trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
        'subsample'            : trial.suggest_float('subsample', 0.6, 1.0),
    }
    cat_cols = [c for c in X.columns if X[c].dtype.name == 'category']
    X_cb = X.copy()
    for c in cat_cols:
        X_cb[c] = X_cb[c].astype(str).replace('nan', 'MISSING')

    gkf = GroupKFold(n_splits=3)
    scores = []
    for tr_idx, val_idx in gkf.split(X_cb, y, groups=groups):
        pool_tr  = cb.Pool(X_cb.iloc[tr_idx],  y.iloc[tr_idx],  cat_features=cat_cols)
        pool_val = cb.Pool(X_cb.iloc[val_idx], y.iloc[val_idx], cat_features=cat_cols)
        m = cb.CatBoostClassifier(**params, random_seed=SEED)
        m.fit(pool_tr, eval_set=pool_val, verbose=False)
        scores.append(average_precision_score(y.iloc[val_idx], m.predict_proba(pool_val)[:,1]))
    return np.mean(scores)


objective_fn = objective_lgb if best_baseline_model == 'lgb' else objective_cb

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective_fn, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest OOF AP (3-fold tuning): {study.best_value:.4f}')
print('Best params:', study.best_params)

## 11. Final Models with Best Params

In [ ]:
if best_baseline_model == 'lgb':
    lgb_tuned_params = {
        'objective'        : 'binary',
        'metric'           : 'average_precision',
        'n_estimators'     : 3000,
        'n_jobs'           : -1,
        'scale_pos_weight' : SPW,
        **study.best_params
    }
    print('=== TUNED LIGHTGBM (5-fold) ===')
    oof_lgb, test_lgb, ap_lgb = run_lgbm(X, y, groups, X_test, lgb_tuned_params, N_FOLDS, SEED)
    oof_cb, test_cb, ap_cb = oof_cb_base, test_cb_base, ap_cb_base
else:
    cb_tuned_params = {
        'iterations'           : 3000,
        'eval_metric'          : 'PRAUC',
        'early_stopping_rounds': 100,
        'loss_function'        : 'Logloss',
        'auto_class_weights'   : 'Balanced',
        'task_type'            : 'CPU',
        **study.best_params
    }
    print('=== TUNED CATBOOST (5-fold) ===')
    oof_cb, test_cb, ap_cb = run_catboost(X, y, groups, X_test, cb_tuned_params, N_FOLDS, SEED)
    oof_lgb, test_lgb, ap_lgb = oof_lgb_base, test_lgb_base, ap_lgb_base

print(f'\nFinal LightGBM OOF AP: {ap_lgb:.4f}')
print(f'Final CatBoost OOF AP: {ap_cb:.4f}')

## 12. Ensemble

In [ ]:
# Grid search ensemble weights on OOF predictions
best_w, best_ens_ap = 0.5, 0.0
results = []

for w_cb in np.arange(0.0, 1.05, 0.05):
    w_lgb = 1.0 - w_cb
    oof_blend = w_lgb * oof_lgb + w_cb * oof_cb
    ens_ap = average_precision_score(y, oof_blend)
    results.append({'w_cb': round(w_cb, 2), 'w_lgb': round(w_lgb, 2), 'oof_ap': ens_ap})
    if ens_ap > best_ens_ap:
        best_ens_ap = ens_ap
        best_w = w_cb

results_df = pd.DataFrame(results)
print('Ensemble weight search:')
print(results_df.sort_values('oof_ap', ascending=False).head(10).to_string(index=False))
print(f'\nBest: CatBoost weight={best_w:.2f}, LGB weight={1-best_w:.2f}  →  OOF AP={best_ens_ap:.4f}')

# Final ensemble predictions
test_ensemble = (1 - best_w) * test_lgb + best_w * test_cb
oof_ensemble  = (1 - best_w) * oof_lgb  + best_w * oof_cb

print(f'\nSummary:')
print(f'  LightGBM alone:  {ap_lgb:.4f}')
print(f'  CatBoost alone:  {ap_cb:.4f}')
print(f'  Ensemble:        {best_ens_ap:.4f}')

## 13. Feature Importance

In [ ]:
lgb_full = lgb.LGBMClassifier(**lgb_tuned_params if best_baseline_model == 'lgb' else lgb_baseline_params,
                               random_state=SEED, verbose=-1)
lgb_full.fit(X, y)

fi = pd.DataFrame({
    'feature'   : feature_cols,
    'gain'      : lgb_full.booster_.feature_importance(importance_type='gain'),
    'split'     : lgb_full.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

print('Top 25 features by gain:')
print(fi.head(25).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 10))
top = fi.head(25)
ax.barh(top['feature'][::-1], top['gain'][::-1], color='#378ADD')
ax.set_title('Top 25 features — LightGBM gain')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 14. Generate Submission

In [ ]:
submission = pd.DataFrame({
    'appearance_id': test_ids,
    'scored_flag'  : test_ensemble
})

assert submission['appearance_id'].nunique() == len(submission), 'Duplicate IDs!'
assert submission['scored_flag'].between(0, 1).all(), 'Out of [0,1]!'
assert submission.isnull().sum().sum() == 0, 'Nulls in submission!'

submission.to_csv('solution.csv', index=False)

print('=== SUBMISSION ===')
print(f'Rows: {len(submission):,}')
print(f'Mean predicted prob: {submission["scored_flag"].mean():.4f}  (train rate: {y.mean():.4f})')
print(f'Min: {submission["scored_flag"].min():.5f}  |  Max: {submission["scored_flag"].max():.4f}')
print('\nsolution.csv saved!')
print(submission.head())

---
## Decision Log

| Decision | Choice | Reason |
|----------|--------|--------|
| Validation | GroupKFold(player_id) | Avg 39.5 appearances per player — StratifiedKFold leaks player info across folds |
| home_club_goals / away_club_goals | KEPT | Present in test set — not pure leakage; team goal count is upper-bound signal |
| goal_diff_abs | KEPT | Present in test set — same reasoning |
| Goalkeeper override | REMOVED | Model already sees is_goalkeeper; manual cap can hurt AP on edge cases |
| Optuna trials | 15 | 12-hour datathon — features >> hyperparams; diminishing returns after ~15 trials |
| Two-model (xG split) | SKIPPED | LightGBM handles nulls natively; splitting halves training data |
| Target encoding | OOF | Smoothed mean encoding for player_id, sub_position, club — computed fold-safe |
| Ensemble | CatBoost + LGB | CatBoost handles high-cardinality categoricals natively; weights tuned on OOF |